# Statistical Methods in Imaging (SMI) Conference, 2026.
# Empowering Large Language Models with Statistics: A Practical Tutorial for Medical Imaging
**Ernest (Khashayar) Namdar, Dominik A. Deniffel, Pascal Tyrrell**

This tutorial focuses on classifying Acute Ischemic Stroke (AIS) from Radiology reports. We will use the `Label` and `Text` columns for a binary classification task.

Several similar pipelines were discussed in our publication:
```bibtex
@inproceedings{10.1117/12.3084682,
author = {Khashayar Namdar and Saeidehsadat Mirjalili and Lauren Erdman and Dominik A. Deniffel and Keith Brunt and Leo Anthony Celi},
title = {{Comparative evaluation of machine learning and large language model pipelines for identifying acute ischemic stroke in radiology reports}},
volume = {13926},
booktitle = {Medical Imaging 2026: Computer-Aided Diagnosis},
editor = {Axel Wism{\"u}ller and Thomas Martin Deserno},
organization = {International Society for Optics and Photonics},
publisher = {SPIE},
pages = {139261S},
keywords = {Stroke, NLP, Machine Learning, Large Language Models},
year = {2026},
doi = {10.1117/12.3084682},
URL = {https://doi.org/10.1117/12.3084682}
}
```

Also, the source for the dataset is:
```bibtex
@article{10.1371/journal.pone.0212778,
    doi = {10.1371/journal.pone.0212778},
    author = {Kim, Chulho AND Zhu, Vivienne AND Obeid, Jihad AND Lenert, Leslie},
    journal = {PLOS ONE},
    publisher = {Public Library of Science},
    title = {Natural language processing and machine learning algorithm to identify brain MRI reports with acute ischemic stroke},
    year = {2019},
    month = {02},
    volume = {14},
    url = {https://doi.org/10.1371/journal.pone.0212778},
    pages = {1-13},
    number = {2},
}
```


## Random Forest Classification
In this section, we apply the exact same nested 10-fold cross-validation pipeline, but substitute LightGBM with a Random Forest classifier using the embeddings.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
import os
import scipy.stats

# Load data
folds_df = pd.read_csv('../data/Folds.csv')
emb_df = pd.read_csv('../data/Embeddings.csv')
df_ais = pd.read_csv('../data/AIS.csv')

# Merge
data = df_ais[['ID', 'Label']].merge(emb_df, on='ID').merge(folds_df, on='ID')

X = data.drop(columns=['ID', 'Label', 'Fold']).values
y = data['Label'].values
folds = data['Fold'].values
ids = data['ID'].values

outer_preds = np.zeros(len(y))

# Nested 10-fold CV
for fold in range(10):
    print(f"--- Outer Fold {fold} ---")
    val_fold = (fold + 1) % 10
    
    test_idx = np.where(folds == fold)[0]
    val_idx = np.where(folds == val_fold)[0]
    train_idx = np.where((folds != fold) & (folds != val_fold))[0]
    
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    X_test, y_test = X[test_idx], y[test_idx]
    
    best_auc = -1
    best_n_estimators = None
    
    for n_trees in [100, 500]:
        model = RandomForestClassifier(n_estimators=n_trees, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        val_preds = model.predict_proba(X_val)[:, 1]
        val_auc = roc_auc_score(y_val, val_preds)
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_n_estimators = n_trees
            
    print(f"Best n_estimators for Fold {fold}: {best_n_estimators} (Val AUC: {best_auc:.4f})")
    
    # Train on train+val with best hyperparameter
    X_train_val = np.concatenate([X_train, X_val])
    y_train_val = np.concatenate([y_train, y_val])
    
    final_model = RandomForestClassifier(n_estimators=best_n_estimators, random_state=42, n_jobs=-1)
    final_model.fit(X_train_val, y_train_val)
    
    # Predict on test
    test_preds = final_model.predict_proba(X_test)[:, 1]
    outer_preds[test_idx] = test_preds

# Save predictions
results_dir = '../results'
os.makedirs(results_dir, exist_ok=True)
preds_df = pd.DataFrame({'ID': ids, 'predictions': outer_preds, 'Label': y})
preds_path = os.path.join(results_dir, 'predictions_rf.csv')
preds_df.to_csv(preds_path, index=False)
print(f"\nPredictions saved to {preds_path}")

# Calculate metrics
fold_aucs = []
for fold in range(10):
    idx = np.where(folds == fold)[0]
    fold_aucs.append(roc_auc_score(y[idx], outer_preds[idx]))

mean_auc = np.mean(fold_aucs)
se = scipy.stats.sem(fold_aucs)
ci = se * scipy.stats.t.ppf((1 + 0.95) / 2., len(fold_aucs)-1)

print(f"Mean AUC: {mean_auc:.4f}")
print(f"95% CI: [{mean_auc - ci:.4f}, {mean_auc + ci:.4f}]")


--- Outer Fold 0 ---
Best n_estimators for Fold 0: 500 (Val AUC: 0.9703)
--- Outer Fold 1 ---
Best n_estimators for Fold 1: 100 (Val AUC: 0.9662)
--- Outer Fold 2 ---
Best n_estimators for Fold 2: 100 (Val AUC: 0.9764)
--- Outer Fold 3 ---
Best n_estimators for Fold 3: 500 (Val AUC: 0.9425)
--- Outer Fold 4 ---
Best n_estimators for Fold 4: 500 (Val AUC: 0.9667)
--- Outer Fold 5 ---
Best n_estimators for Fold 5: 500 (Val AUC: 0.9746)
--- Outer Fold 6 ---
Best n_estimators for Fold 6: 500 (Val AUC: 0.9631)
--- Outer Fold 7 ---
Best n_estimators for Fold 7: 500 (Val AUC: 0.9600)
--- Outer Fold 8 ---
Best n_estimators for Fold 8: 500 (Val AUC: 0.9626)
--- Outer Fold 9 ---
Best n_estimators for Fold 9: 500 (Val AUC: 0.9749)

Predictions saved to ../results/predictions_rf.csv
Mean AUC: 0.9659
95% CI: [0.9595, 0.9724]
